# 3. Incidents, Investigation, and Automated Response

## From alerts to incidents

A single attack generates multiple alerts across different products. The SIEM **correlates** them into a single **incident** — the unit of work for a SOC analyst.

```
Alert: Brute force sign-in (alice)           ─┐
Alert: Sign-in from suspicious location       ├── Incident: Compromised account (alice)
Alert: Lateral movement from alice's laptop   ─┘
```

In [ ]:
import httpx, json

SIEM = 'http://localhost:8000'

# View all incidents
incidents = httpx.get(f'{SIEM}/incidents').json()
print(f'=== Open Incidents ({len(incidents)}) ===\n')
for inc in incidents:
    sev = {'Critical': '🟣', 'High': '🔴', 'Medium': '🟡', 'Low': '🟢'}.get(inc['severity'], '⬜')
    alerts = json.loads(inc['alert_ids']) if isinstance(inc['alert_ids'], str) else inc['alert_ids']
    entities = json.loads(inc['entities']) if isinstance(inc['entities'], str) else inc['entities']
    print(f'{sev} {inc["id"]} [{inc["status"]}]')
    print(f'   Title: {inc["title"]}')
    print(f'   Alerts: {len(alerts)}  |  Entities: {entities}')
    print(f'   Assigned: {inc["assigned_to"] or "Unassigned"}')
    print()

## Incident investigation workflow

The SC-200 exam tests your ability to **investigate** incidents. Here's the workflow:

1. **Triage** — assess severity, check if it's a true positive
2. **Investigate** — drill into alerts, examine entities, check timeline
3. **Contain** — isolate affected resources, disable accounts
4. **Remediate** — remove threat, patch vulnerability
5. **Close** — classify and document findings

Let's investigate the highest-severity incident:

In [ ]:
# Pick the first High-severity incident
high_incidents = [i for i in incidents if i['severity'] == 'High']
if high_incidents:
    target = high_incidents[0]
    print(f'=== Investigating: {target["id"]} ===')
    
    # Get full incident details
    details = httpx.get(f'{SIEM}/incidents/{target["id"]}').json()
    
    print(f'\nTitle: {details["title"]}')
    print(f'Severity: {details["severity"]}')
    print(f'Status: {details["status"]}')
    print(f'Created: {details["created_at"]}')
    
    print(f'\n--- Related Alerts ({len(details["alerts"])}) ---')
    for alert in details['alerts']:
        print(f'  🚨 {alert["title"]}')
        print(f'     Tactic: {alert["tactic"]}  |  Severity: {alert["severity"]}')
        evidence = json.loads(alert['evidence']) if isinstance(alert['evidence'], str) else alert['evidence']
        if evidence:
            print(f'     Evidence sample: {json.dumps(evidence[0], indent=6)[:200]}')
        print()

In [ ]:
# Step 1: Triage — assign to analyst, change status
if high_incidents:
    inc_id = high_incidents[0]['id']
    
    print(f'=== Step 1: Triage {inc_id} ===')
    r = httpx.patch(f'{SIEM}/incidents/{inc_id}', json={
        'status': 'Active',
        'assigned_to': 'soc-analyst-1@contoso.com',
        'comment': 'Triaged: legitimate brute force attack on alice. Investigating.',
    })
    print(f'Status: {r.json()["status"]}')
    
    # Verify
    updated = httpx.get(f'{SIEM}/incidents/{inc_id}').json()
    print(f'Assigned to: {updated["assigned_to"]}')
    print(f'Status: {updated["status"]}')
    comments = json.loads(updated['comments']) if isinstance(updated['comments'], str) else updated['comments']
    print(f'Comments: {comments}')

In [ ]:
# Step 2: Investigate — query related activity
if high_incidents:
    entities = json.loads(high_incidents[0]['entities']) if isinstance(high_incidents[0]['entities'], str) else high_incidents[0]['entities']
    
    if 'UserPrincipalName' in entities:
        user = entities['UserPrincipalName']
        print(f'=== Step 2: Investigate user {user} ===\n')
        
        # All sign-in activity for this user
        print('--- Sign-in activity ---')
        r = httpx.post(f'{SIEM}/query', json={
            'table_name': 'SigninLogs',
            'filter': {'UserPrincipalName': user},
            'aggregate_by': 'ResultType',
        })
        for row in r.json()['results']:
            print(f'  {row["group_key"]}: {row["count"]} events')
        
        # Check for endpoint activity
        print(f'\n--- Endpoint activity for {user.split("@")[0]} ---')
        r = httpx.post(f'{SIEM}/query', json={
            'table_name': 'DeviceEvents',
            'filter': {'AccountName': user.split('@')[0]},
            'limit': 10,
        })
        for log in r.json()['results']:
            suspicious = '⚠️' if log.get('FileName') in ('mimikatz.exe', 'psexec.exe', 'cmd.exe') else '  '
            print(f'  {suspicious} {log["DeviceName"]}: {log["FileName"]} ({log["ActionType"]})')

In [ ]:
# Step 3: Automated response — run playbooks
if high_incidents:
    inc_id = high_incidents[0]['id']
    print(f'=== Step 3: Run playbooks for {inc_id} ===\n')
    
    r = httpx.post(f'{SIEM}/playbooks/run/{inc_id}')
    result = r.json()
    
    if result['playbooks_executed']:
        for pb in result['playbooks_executed']:
            print(f'🤖 Playbook: {pb["playbook"]}')
            for i, action in enumerate(pb['actions'], 1):
                print(f'   Step {i}: ✅ {action["description"]}')
            print()
    else:
        print('No matching playbooks found.')
    
    print('\n💡 In real Sentinel, playbooks are Logic Apps that call Azure APIs:')
    print('   - Disable user: Microsoft Graph API')
    print('   - Isolate device: Defender for Endpoint API')
    print('   - Block IP: Azure Firewall API')
    print('   - Send notification: Teams webhook / Email')

In [ ]:
# Step 4: Close incident with classification
if high_incidents:
    inc_id = high_incidents[0]['id']
    print(f'=== Step 4: Close incident {inc_id} ===\n')
    
    r = httpx.patch(f'{SIEM}/incidents/{inc_id}', json={
        'status': 'Closed',
        'classification': 'TruePositive',
        'comment': 'Confirmed brute force attack. User account disabled. Password reset required. Playbook executed.',
    })
    
    final = httpx.get(f'{SIEM}/incidents/{inc_id}').json()
    print(f'Status: {final["status"]}')
    print(f'Classification: {final["classification"]}')
    comments = json.loads(final['comments']) if isinstance(final['comments'], str) else final['comments']
    for c in comments:
        print(f'Comment: {c["text"]}')
    
    print('\n--- Incident classifications ---')
    print('  TruePositive — confirmed attack, action taken')
    print('  BenignPositive — real activity, not malicious (e.g., pen test)')
    print('  FalsePositive — detection was wrong, tune the rule')
    print('  Undetermined — not enough evidence')

## Automation rules vs playbooks

| | Automation rules | Playbooks |
|-|-----------------|----------|
| **What** | Lightweight if/then logic | Full Logic App workflows |
| **Trigger** | Incident creation/update, alert creation | Called by automation rules or manually |
| **Actions** | Change severity, assign, run playbook, close | Any Azure/external API call |
| **Code** | No-code (portal UI) | Low-code (Logic Apps designer) |
| **Use case** | Triage automation, suppress noise | Complex response workflows |

### SC-200 exam tip

- **Automation rules** decide *when* to run a playbook.
- **Playbooks** define *what actions* to take.
- Automation rules can also suppress, re-assign, and close incidents without playbooks.
- Maximum **512 automation rules** per workspace.

---
## What you built

You now have a working mini-SIEM with:
- ✅ Data ingestion from 4+ sources
- ✅ Query engine with filtering and aggregation
- ✅ 6+ analytics rules with MITRE ATT&CK mapping
- ✅ Alert → Incident correlation
- ✅ Full investigation workflow (triage → investigate → respond → close)
- ✅ Automated playbook execution

This is exactly what Sentinel does — at massive scale with KQL, Logic Apps, and ML.

**Next lab**: [02 — Incident Response](../../02-incident-response/)